In [17]:
debut_test = pd.Timestamp("2016-02-29")
fin_test   = pd.Timestamp("2016-03-27")

train = df[df["date"] < debut_test]
test  = df[(df["date"] >= debut_test) & (df["date"] <= fin_test)].copy()

print("Train :", train["date"].min().date(), "→", train["date"].max().date())
print("Test  :", test["date"].min().date(), "→", test["date"].max().date())

Train : 2011-01-29 → 2016-02-28
Test  : 2016-02-29 → 2016-03-27


In [19]:
# ---------- 1. Découpage train / test (Test 1) ----------
debut_test = pd.Timestamp("2016-02-29")
fin_test   = pd.Timestamp("2016-03-27")

train = df[df["date"] < debut_test]
test  = df[(df["date"] >= debut_test) & (df["date"] <= fin_test)].copy()

# ---------- 2. Baseline 1 : moyenne des 28 derniers jours ----------
p28 = train[train["date"] >= debut_test - pd.Timedelta(days=28)]
moyenne = p28.groupby("item_id")["sales"].mean()
test["prevu_moy28"] = test["item_id"].map(moyenne).fillna(0)

# ---------- 3. Baseline 2 : même jour, semaine dernière ----------
p7 = train[train["date"] >= debut_test - pd.Timedelta(days=7)]
p7 = p7[["item_id", "weekday", "sales"]].rename(columns={"sales": "prevu_sem"})
test = test.merge(p7, on=["item_id", "weekday"], how="left")
test["prevu_sem"] = test["prevu_sem"].fillna(0)

# ---------- 4. Baseline 3 : moyenne du même jour sur 8 semaines ----------
p56 = train[train["date"] >= debut_test - pd.Timedelta(days=56)]
moy_jour = (p56.groupby(["item_id", "weekday"])["sales"].mean()
               .reset_index().rename(columns={"sales": "prevu_sem8"}))
test = test.merge(moy_jour, on=["item_id", "weekday"], how="left")
test["prevu_sem8"] = test["prevu_sem8"].fillna(0)

# ---------- 5. Résultats ----------
totaux = test.groupby("item_id")[["sales", "prevu_moy28", "prevu_sem", "prevu_sem8"]].sum()

for col in ["prevu_moy28", "prevu_sem", "prevu_sem8"]:
    print(col,
          "| WAPE jour :", round(wape(test["sales"], test[col]) * 100, 1), "%",
          "| WAPE total 28 j :", round(wape(totaux["sales"], totaux[col]) * 100, 1), "%")

prevu_moy28 | WAPE jour : 75.3 % | WAPE total 28 j : 27.8 %
prevu_sem | WAPE jour : 86.9 % | WAPE total 28 j : 34.1 %
prevu_sem8 | WAPE jour : 75.6 % | WAPE total 28 j : 29.7 %


In [20]:
# ---------- 1. Découpage train / test (Test 1) ----------
debut_test = pd.Timestamp("2016-03-28")
fin_test   = pd.Timestamp("2016-04-24")

train = df[df["date"] < debut_test]
test  = df[(df["date"] >= debut_test) & (df["date"] <= fin_test)].copy()

# ---------- 2. Baseline 1 : moyenne des 28 derniers jours ----------
p28 = train[train["date"] >= debut_test - pd.Timedelta(days=28)]
moyenne = p28.groupby("item_id")["sales"].mean()
test["prevu_moy28"] = test["item_id"].map(moyenne).fillna(0)

# ---------- 3. Baseline 2 : même jour, semaine dernière ----------
p7 = train[train["date"] >= debut_test - pd.Timedelta(days=7)]
p7 = p7[["item_id", "weekday", "sales"]].rename(columns={"sales": "prevu_sem"})
test = test.merge(p7, on=["item_id", "weekday"], how="left")
test["prevu_sem"] = test["prevu_sem"].fillna(0)

# ---------- 4. Baseline 3 : moyenne du même jour sur 8 semaines ----------
p56 = train[train["date"] >= debut_test - pd.Timedelta(days=56)]
moy_jour = (p56.groupby(["item_id", "weekday"])["sales"].mean()
               .reset_index().rename(columns={"sales": "prevu_sem8"}))
test = test.merge(moy_jour, on=["item_id", "weekday"], how="left")
test["prevu_sem8"] = test["prevu_sem8"].fillna(0)

# ---------- 5. Résultats ----------
totaux = test.groupby("item_id")[["sales", "prevu_moy28", "prevu_sem", "prevu_sem8"]].sum()

for col in ["prevu_moy28", "prevu_sem", "prevu_sem8"]:
    print(col,
          "| WAPE jour :", round(wape(test["sales"], test[col]) * 100, 1), "%",
          "| WAPE total 28 j :", round(wape(totaux["sales"], totaux[col]) * 100, 1), "%")

prevu_moy28 | WAPE jour : 74.5 % | WAPE total 28 j : 26.8 %
prevu_sem | WAPE jour : 88.8 % | WAPE total 28 j : 36.2 %
prevu_sem8 | WAPE jour : 75.0 % | WAPE total 28 j : 27.1 %


In [ ]:
test = test.drop(columns="prevu_sem", errors="ignore")

derniere_semaine = train[train["date"] >= debut_test - pd.Timedelta(days=7)]
derniere_semaine = derniere_semaine[["item_id", "weekday", "sales"]].rename(columns={"sales": "prevu_sem"})

test = test.merge(derniere_semaine, on=["item_id", "weekday"], how="left")
test["prevu_sem"] = test["prevu_sem"].fillna(0)

print("WAPE  :", round(wape(test["sales"], test["prevu_sem"]) * 100, 1), "%")
print("Biais :", round(biais(test["sales"], test["prevu_sem"]) * 100, 1), "%")

WAPE  : 86.9 %
Biais : -4.5 %


In [ ]:
test = test.drop(columns="prevu_sem8", errors="ignore")

huit_semaines = train[train["date"] >= debut_test - pd.Timedelta(days=56)]

moy_jour = (
    huit_semaines.groupby(["item_id", "weekday"])["sales"]
    .mean()
    .reset_index()
    .rename(columns={"sales": "prevu_sem8"})
)

test = test.merge(moy_jour, on=["item_id", "weekday"], how="left")
test["prevu_sem8"] = test["prevu_sem8"].fillna(0)

print("WAPE  :", round(wape(test["sales"], test["prevu_sem8"]) * 100, 1), "%")
print("Biais :", round(biais(test["sales"], test["prevu_sem8"]) * 100, 1), "%")

WAPE  : 75.6 %
Biais : -3.8 %


In [ ]:
derniers_28j = train[train["date"] >= debut_test - pd.Timedelta(days=28)]

moyenne = derniers_28j.groupby("item_id")["sales"].mean()

test["prevu_moy28"] = test["item_id"].map(moyenne).fillna(0)

print("WAPE  :", round(wape(test["sales"], test["prevu_moy28"]) * 100, 1), "%")
print("Biais :", round(biais(test["sales"], test["prevu_moy28"]) * 100, 1), "%")

WAPE  : 75.3 %
Biais : -1.9 %


In [ ]:
def wape(vrai, prevu):
    return np.abs(vrai - prevu).sum() / vrai.sum()

def biais(vrai, prevu):
    return (prevu.sum() - vrai.sum()) / vrai.sum()

# Test avec l'exemple d'hier
vrai  = np.array([10, 0, 5, 5])
prevu = np.array([ 8, 1, 5, 7])

print("WAPE  :", round(wape(vrai, prevu) * 100, 1), "%")
print("Biais :", round(biais(vrai, prevu) * 100, 1), "%")

WAPE  : 25.0 %
Biais : 5.0 %


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/processed/sales_ca1_clean.parquet")
print(df.shape)

(4788267, 17)
